# Phase 5: Model Validation & Monitoring
This notebook executes banking-grade model validation. We compute ROC-AUC, KS, Gini, and Brier scores, plot calibration curves, and evaluate temporal stability using Population Stability Index (PSI) and Characteristic Stability Index (CSI) against the Out-of-Time validation cohort.


In [ ]:
import pandas as pd
import os
import sys

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from woe_binning import WoEBinning
from scorecard import ScorecardModel
from validation import ModelValidator
from stability import StabilityMonitor


## 1. Load Model Predictions
We load scored datasets to run performance checks.


In [ ]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()

woe_model = WoEBinning(target_col='target')
woe_model.fit_all(train_df, [
    'loan_amnt', 'annual_inc', 'dti', 'revol_util', 
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 
    'pub_rec', 'pub_rec_bankruptcies', 'credit_history_age',
    'emp_length'
], ['home_ownership', 'purpose'])

train_woe = woe_model.transform(train_df)
oot_woe = woe_model.transform(oot_df)
selected_woe_cols = [f'{col}_woe' for col in woe_model.selected_features]

scorecard = ScorecardModel()
scorecard.fit(train_woe[selected_woe_cols], train_woe['target'])
scorecard.build_scorecard_table(woe_model.mappings)

train_scores = scorecard.predict_score(train_woe)
oot_scores = scorecard.predict_score(oot_woe)
oot_pds = scorecard.predict_pd(oot_woe)


## 2. Credit Performance Curves
Plot ROC and KS curves to evaluate risk differentiation.


In [ ]:
y_true_oot = oot_df['target'].values
ModelValidator.plot_roc_curve(y_true_oot, oot_pds, 'ROC Curve - Out-of-Time Validation')
ModelValidator.plot_ks_curve(y_true_oot, oot_pds, 'KS Separation Curve - Out-of-Time Validation')


## 3. Decile Analysis
Verify that default rates decrease monotonically as credit scores increase (which corresponds to higher deciles of score or lower deciles of default probability).


In [ ]:
decile_df = ModelValidator.generate_decile_analysis(y_true_oot, oot_pds)
print(decile_df[['decile', 'total_loans', 'defaults', 'default_rate', 'ks_stat', 'lift']])


## 4. Temporal Stability & Drift: PSI & CSI
Calculate Population Stability Index (PSI) and Characteristic Stability Index (CSI) between In-Time training and Out-of-Time validation cohorts.


In [ ]:
psi_val, psi_details = StabilityMonitor.calculate_psi(train_scores, oot_scores)
print(f'Total Portfolio Credit Score PSI: {psi_val:.4f}')
if psi_val < 0.10:
    print('Status: STABLE (No action required)')
elif psi_val < 0.25:
    print('Status: MODERATE SHIFT (Monitor closely)')
else:
    print('Status: SIGNIFICANT DRIFT (Requires retraining)')

csi_vals, _ = StabilityMonitor.calculate_csi(train_df, oot_df, woe_model.mappings)
print('\nTop CSI Variables (Characteristic Drift):')
for col, val in sorted(csi_vals.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f' - {col}: {val:.4f}')
